# Sentinel-1 / Tessa's LiDAR Patch Collocation (UTM Zone 8N)

**Purpose:** produce a fully-collocated Sentinel-1 dataset matched against Tessa's original LiDAR patches (`lidar_patches_tuk_tessa`), following Michel Tsamados's `patching_sentinel1.ipynb` methodology exactly.

**Why this notebook exists separately from `03_sentinel1_preprocessing.ipynb`:**

The existing calibrated Sentinel-1 products (`t0.tif`-`t7.tif` in `raw_data/tuk_sentinel1_diy_corrected/`) were warped to **EPSG:6931** (the polar EASE-Grid), matching this project's *own* LiDAR patches. Tessa's original patches are instead stored in **EPSG:32608 (UTM Zone 8N)**. Collocating across two different CRSs forces a real reprojection, and because Tuktoyaktuk sits at high latitude, UTM 8N's "north" and EASE-Grid's "north" differ by close to 45 degrees here -- so a 256m x 256m LiDAR patch reprojects into a rotated shape whose axis-aligned bounding box inflates to roughly 360m x 360m (a factor of ~sqrt(2), consistent with a ~45-degree rotation).

Michel's own pipeline avoids this entirely by deriving `dst_crs` directly from whichever LiDAR patches are in use (`dst_crs = ref.crs`, read from the first LiDAR patch file) -- for him, that's Tessa's native UTM 8N. This notebook replicates that: a **second** Sentinel-1 calibration pass, targeting UTM 8N specifically for use with Tessa's patches, followed by Michel's stricter patch-matching function.

**Cost warning:** the calibration step below repeats the same expensive GCP-warp + radiometric calibration work as the original `build_products()` run in notebook 03 -- expect roughly 4-5 hours for all 8 products, same order of magnitude as before. It reads the same raw SAFE downloads already on disk; nothing needs re-downloading.

## 1. Imports

In [ ]:
import os
import json
import glob as glob_module
import xml.etree.ElementTree as ET

import numpy as np
import rasterio
from rasterio.windows import Window, from_bounds
from rasterio.warp import transform_bounds, calculate_default_transform, reproject, Resampling
from scipy.interpolate import RectBivariateSpline

## 2. Configuration

Points at data already on the project store -- no new downloads needed. `DIY_CORRECTED_UTM_DIR` is a **new** directory, kept separate from the existing EPSG:6931 output (`tuk_sentinel1_diy_corrected/`), so nothing already built gets overwritten.

In [ ]:
REPO_DIR = '/cs/student/project_msc/2025/aibh/jiayiche'
REGION = 'tuk'

RAW_S1_DOWNLOADS_DIR = f'{REPO_DIR}/raw_data/{REGION}_sentinel1_downloads'
DIY_CORRECTED_UTM_DIR = f'{REPO_DIR}/raw_data/{REGION}_sentinel1_diy_corrected_utm8n'
os.makedirs(DIY_CORRECTED_UTM_DIR, exist_ok=True)

lidar_patches_tessa_dir = f'{REPO_DIR}/input_data/lidar_patches_{REGION}_tessa'

patch_size = 256
s1_patch_size = int(round(patch_size / 10))  # 26 px @ 10m IW resolution

out_s1_dir = f'{REPO_DIR}/input_data/s1_patches_{REGION}_tessa_utm8n'
os.makedirs(out_s1_dir, exist_ok=True)

print('s1_patch_size:', s1_patch_size)

## 3. Determine `dst_crs` from Tessa's patches -- the key fix

This is the one line that actually solves the CRS-mismatch problem: read the CRS straight off one of Tessa's own LiDAR patches, rather than hardcoding EPSG:6931. Since Tessa's patches are in UTM Zone 8N, this should print `EPSG:32608`.

In [ ]:
with rasterio.open(sorted(glob_module.glob(os.path.join(lidar_patches_tessa_dir, 'lidar_patch_*.tif')))[0]) as ref:
    dst_crs = ref.crs
print('dst_crs (from Tessa patches):', dst_crs)

## 4. Find the raw SAFE product directories

Unmodified from Michel's original `find_safe_dirs` -- checked against this project's actual on-disk layout (`tuk_sentinel1_downloads/X.SAFE/X.SAFE/`, one wrapper folder around the real SAFE folder from how the zips were extracted) and confirmed to match correctly as-is.

In [ ]:
def find_safe_dirs(downloads_dir):
    return sorted(glob_module.glob(os.path.join(downloads_dir, '*', '*.SAFE')))

safe_dirs = find_safe_dirs(RAW_S1_DOWNLOADS_DIR)
print(f'Found {len(safe_dirs)} SAFE products')
for d in safe_dirs:
    print(' ', d)

**Check before continuing:** confirm this printed **8** -- matching the verified, coverage-filtered product set from notebook 03. If it shows a different number, stop and investigate before running the expensive calibration step below.

## 5. Calibration and warp helper functions (Michel's code, unchanged)

- `find_pol_files`: locates the measurement GeoTIFF and calibration XML for a given polarization inside a SAFE folder.
- `build_calibration_lut`: parses the sparse calibration XML into a full-resolution sigma-nought lookup grid via bilinear interpolation.
- `compute_gcp_warp_grid`: computes the destination transform/size **once** per product (from VV), reused for VH, guaranteeing both polarizations land on an identical pixel grid.
- `calibrate_and_warp_s1_band`: converts digital numbers to sigma-nought, then GCP-warps onto the shared grid. Streams the warp directly to disk via `rasterio.band()` rather than holding a full destination array in memory.
- `merge_pol_bands_to_geotiff`: stacks the separately-warped VV and VH files into one 2-band GeoTIFF.
- `diy_correct_s1_products`: runs the full pipeline over every SAFE product.

In [ ]:
def find_pol_files(safe_dir, pol):
    meas = glob_module.glob(os.path.join(safe_dir, "measurement", f"*-{pol}-*.tiff"))
    cal  = glob_module.glob(os.path.join(safe_dir, "annotation", "calibration", f"calibration-*-{pol}-*.xml"))
    if not meas:
        raise FileNotFoundError(f"No {pol} measurement file found in {safe_dir}")
    if not cal:
        raise FileNotFoundError(f"No {pol} calibration XML found in {safe_dir}")
    return meas[0], cal[0]


def build_calibration_lut(calibration_xml_path, image_shape):
    tree = ET.parse(calibration_xml_path)
    root = tree.getroot()
    lines, pixel_grid, sigma_rows = [], None, []
    for vec in root.find("calibrationVectorList").findall("calibrationVector"):
        line = int(vec.find("line").text)
        pixels = np.array([int(v) for v in vec.find("pixel").text.split()])
        sigma = np.array([float(v) for v in vec.find("sigmaNought").text.split()])
        lines.append(line)
        if pixel_grid is None:
            pixel_grid = pixels
        sigma_rows.append(sigma)
    lines = np.array(lines, dtype=np.float64)
    pixel_grid = pixel_grid.astype(np.float64)
    sigma_grid = np.vstack(sigma_rows)
    interp = RectBivariateSpline(lines, pixel_grid, sigma_grid, kx=1, ky=1)
    H, W = image_shape
    return interp(np.arange(H), np.arange(W))


def compute_gcp_warp_grid(measurement_tiff, dst_crs, dst_resolution=10.0):
    with rasterio.open(measurement_tiff) as src:
        gcps, gcp_crs = src.gcps
        dst_transform, dst_width, dst_height = calculate_default_transform(
            gcp_crs, dst_crs, src.width, src.height,
            gcps=gcps, resolution=(dst_resolution, dst_resolution),
        )
    return dst_transform, dst_width, dst_height


def calibrate_and_warp_s1_band(measurement_tiff, calibration_xml, dst_crs, out_path, warp_grid):
    import gc
    dst_transform, dst_width, dst_height = warp_grid
    with rasterio.open(measurement_tiff) as src:
        dn = src.read(1).astype(np.float32)
        gcps, gcp_crs = src.gcps
    sigma_lut = build_calibration_lut(calibration_xml, dn.shape).astype(np.float32)
    with np.errstate(divide="ignore", invalid="ignore"):
        sigma0 = np.where(sigma_lut > 0, dn ** 2 / sigma_lut ** 2, np.nan).astype(np.float32)
    del dn, sigma_lut
    gc.collect()
    profile = {
        "driver": "GTiff", "dtype": "float32", "count": 1,
        "height": dst_height, "width": dst_width,
        "crs": dst_crs, "transform": dst_transform, "nodata": np.nan,
    }
    with rasterio.open(out_path, "w", **profile) as dst:
        reproject(
            source=sigma0,
            destination=rasterio.band(dst, 1),
            gcps=gcps,
            src_crs=gcp_crs,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=Resampling.bilinear,
            src_nodata=np.nan,
            dst_nodata=np.nan,
        )
    del sigma0
    gc.collect()
    return out_path


def merge_pol_bands_to_geotiff(vv_path, vh_path, out_path):
    with rasterio.open(vv_path) as vv_src, rasterio.open(vh_path) as vh_src:
        if vv_src.shape != vh_src.shape:
            raise ValueError(f"VV/VH warped shapes differ ({vv_src.shape} vs {vh_src.shape})")
        vv = vv_src.read(1); vh = vh_src.read(1)
        meta = vv_src.meta.copy(); meta.update(count=2)
        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(vv, 1); dst.write(vh, 2)
    return out_path


def diy_correct_s1_products(safe_dirs, dst_crs, out_dir, dst_resolution=10.0):
    os.makedirs(out_dir, exist_ok=True)
    out_paths, attrs_paths = [], []
    for i, safe_dir in enumerate(sorted(safe_dirs)):
        print(f"Processing product {i+1}/{len(safe_dirs)}: {os.path.basename(safe_dir)}")
        vv_meas, vv_cal = find_pol_files(safe_dir, "vv")
        vh_meas, vh_cal = find_pol_files(safe_dir, "vh")
        warp_grid = compute_gcp_warp_grid(vv_meas, dst_crs, dst_resolution)
        tmp_vv = os.path.join(out_dir, f"_tmp_vv_{i}.tif")
        tmp_vh = os.path.join(out_dir, f"_tmp_vh_{i}.tif")
        calibrate_and_warp_s1_band(vv_meas, vv_cal, dst_crs, tmp_vv, warp_grid)
        calibrate_and_warp_s1_band(vh_meas, vh_cal, dst_crs, tmp_vh, warp_grid)
        final_path = os.path.join(out_dir, f"t{i}.tif")
        merge_pol_bands_to_geotiff(tmp_vv, tmp_vh, final_path)
        os.remove(tmp_vv); os.remove(tmp_vh)
        out_paths.append(final_path)
        import gc; gc.collect()
        attrs_paths.append(None)  # metadata sidecar naming doesn't match this project's download convention
        print(f"  -> {final_path}")
    return out_paths, attrs_paths

## 6. Run the calibration + warp (slow -- ~4-5 hours for 8 products)

This is the expensive step. Safe to let run unattended once Step 7's early check below confirms the first product looks correct -- consistent with the same monitoring approach used in notebook 03 (`ls -lh` on the output directory to watch progress).

In [ ]:
diy_paths, diy_attrs = diy_correct_s1_products(safe_dirs, dst_crs, DIY_CORRECTED_UTM_DIR, dst_resolution=10.0)

## 7. Verify before proceeding to the expensive match step

Per Michel's own notebook structure: always confirm calibration actually produced valid data before running the slow patch-matching step. A real, non-zero valid fraction (expect roughly 0.4-0.6, similar to the EPSG:6931 run's `nonzero_frac` results) means it worked.

In [ ]:
for p in diy_paths:
    with rasterio.open(p) as src:
        small = src.read(1, out_shape=(1, src.height // 20, src.width // 20))
        print(f"{os.path.basename(p)}: valid fraction = {np.mean(~np.isnan(small)):.4f}")

print('\\nOnly proceed to the match step below if every file above shows a real, non-zero valid fraction.')

## 8. Patch-matching functions (Michel's stricter collocation logic)

Key differences from this project's original `collocate()`:
- Enforces an **exact** target window size (`s1_patch_size`) -- any product whose reprojected window doesn't match exactly is rejected.
- Rejects the **entire patch** (all timesteps) if any single product fails, rather than writing a partial patch.
- Explicit `max_nan_frac` threshold (default 2%) -- rejects a patch if too many pixels are NaN, even if not completely empty.

Since `dst_crs` now matches Tessa's patches' own CRS exactly (both UTM 8N), `transform_bounds()` inside this function is a no-op here -- no rotation, no inflation, so windows should come out at the expected fixed 26x26 size rather than the 36x36 seen when mixing CRSs.

In [ ]:
def build_s1_products_from_corrected(geotiff_paths, attrs_jsons=None):
    products = []
    for i, path in enumerate(geotiff_paths):
        src = rasterio.open(path)
        attrs = None
        if attrs_jsons and i < len(attrs_jsons) and attrs_jsons[i] and os.path.exists(attrs_jsons[i]):
            with open(attrs_jsons[i]) as jf:
                attrs = json.load(jf)
        products.append({"src": src, "crs": src.crs, "transform": src.transform,
                          "height": src.height, "width": src.width, "attrs": attrs})
    if not products:
        raise ValueError("No Sentinel-1 products loaded -- check geotiff_paths.")
    return products


def close_products(products):
    for p in products:
        p["src"].close()


def extract_lidar_matched_s1_patches(lidar_patches_dir, sentinel1_products, s1_patch_size,
                                      out_s1_dir, pattern="lidar_patch_*.tif", max_nan_frac=0.02):
    lidar_paths = sorted(glob_module.glob(os.path.join(lidar_patches_dir, pattern)))
    print(f"Found {len(lidar_paths)} existing LiDAR patches to match against "
          f"{len(sentinel1_products)} Sentinel-1 product(s).")

    n_written, n_skipped, n_skipped_nan = 0, 0, 0

    for idx, lp in enumerate(lidar_paths):
        if idx % 100 == 0:
            print(f"  ...processed {idx}/{len(lidar_paths)} "
                  f"(written: {n_written}, skipped: {n_skipped}, skipped-NaN: {n_skipped_nan})")

        patch_id = os.path.splitext(os.path.basename(lp))[0].split("_")[-1]

        with rasterio.open(lp) as lsrc:
            lidar_bounds = lsrc.bounds
            lidar_crs = lsrc.crs

        s1_patches, s1_transforms = [], []
        ok = True
        has_too_much_nan = False
        for prod in sentinel1_products:
            try:
                s1_bounds = transform_bounds(lidar_crs, prod["crs"], *lidar_bounds, densify_pts=21)
                window = from_bounds(*s1_bounds, transform=prod["transform"]).round_offsets().round_lengths()
                r0, c0 = int(window.row_off), int(window.col_off)
                hh, ww = int(window.height), int(window.width)

                if (hh, ww) != (s1_patch_size, s1_patch_size):
                    ok = False; break
                if r0 < 0 or c0 < 0:
                    ok = False; break
                if r0 + s1_patch_size > prod["height"] or c0 + s1_patch_size > prod["width"]:
                    ok = False; break

                read_window = Window(c0, r0, s1_patch_size, s1_patch_size)
                patch = prod["src"].read(window=read_window)
                if patch.shape[1:] != (s1_patch_size, s1_patch_size):
                    ok = False; break

                nan_frac = float(np.mean(np.isnan(patch)))
                if nan_frac > max_nan_frac:
                    has_too_much_nan = True
                    break

                s1_patches.append(patch)
                s1_transforms.append(rasterio.windows.transform(read_window, prod["transform"]))
            except Exception:
                ok = False
                break

        if has_too_much_nan:
            n_skipped_nan += 1
            continue

        if not ok or len(s1_patches) != len(sentinel1_products):
            n_skipped += 1
            continue

        patch_dir = os.path.join(out_s1_dir, f"s1_patch_{patch_id}")
        os.makedirs(patch_dir, exist_ok=True)

        attrs_list = []
        for ti, (prod, patch, tr) in enumerate(zip(sentinel1_products, s1_patches, s1_transforms)):
            meta = {
                "driver": "GTiff", "count": patch.shape[0],
                "height": s1_patch_size, "width": s1_patch_size,
                "dtype": "float32", "crs": prod["crs"], "transform": tr,
            }
            with rasterio.open(os.path.join(patch_dir, f"t{ti}.tif"), "w", **meta) as dst:
                dst.write(patch.astype(np.float32))
            attrs_list.append(prod.get("attrs"))

        with open(os.path.join(patch_dir, "attrs.json"), "w") as jf:
            json.dump(attrs_list, jf, indent=2)

        n_written += 1

    print(f"Done. Matched: {n_written}, skipped (out of bounds/wrong size): {n_skipped}, "
          f"skipped (too much NaN, >{max_nan_frac:.0%}): {n_skipped_nan}")
    return n_written, n_skipped

## 9. Run the match (slow -- ~20-30 min for the full 1676-patch set)

In [ ]:
products = build_s1_products_from_corrected(diy_paths, attrs_jsons=diy_attrs)
extract_lidar_matched_s1_patches(lidar_patches_tessa_dir, products, s1_patch_size, out_s1_dir)
close_products(products)

## 10. Verify the final output

Confirm the window size came out as the expected fixed 26x26 (no CRS-mismatch inflation this time), and check the distribution of how many timesteps each written patch actually has.

In [ ]:
sample_patches = sorted(glob_module.glob(os.path.join(out_s1_dir, 's1_patch_*')))
print(f'Total patches written: {len(sample_patches)}')

if sample_patches:
    with rasterio.open(os.path.join(sample_patches[0], 't0.tif')) as src:
        print('Sample patch shape:', src.shape, '(expect 26x26, not 36x36)')

from collections import Counter
counts = Counter(len(glob_module.glob(os.path.join(p, 't*.tif'))) for p in sample_patches)
print('Distribution of timesteps per patch:', dict(sorted(counts.items())))

## 11. Visual verification -- `show_multiple_matched_patches` (Michel's code, unchanged)

Shows several random matched patches side by side: LiDAR elevation anomaly, LiDAR valid mask, and every Sentinel-1 timestep/band (converted to dB), with a constant color scale per band across all sampled patches so comparisons are meaningful. Re-run the last line for a fresh random sample each time.

In [ ]:
import random
import matplotlib.pyplot as plt

def show_multiple_matched_patches(lidar_patches_dir, out_s1_dir, n_samples=5, seed=None, use_db=True,
                                   pol_labels=("VV", "VH")):
    sample_dirs = sorted(glob_module.glob(os.path.join(out_s1_dir, "s1_patch_*")))
    if not sample_dirs:
        print(f"No matched patches found in {out_s1_dir}")
        return

    rng = random.Random(seed)
    n_samples = min(n_samples, len(sample_dirs))
    picks = rng.sample(sample_dirs, n_samples)
    pids = [os.path.basename(p).split("_")[-1] for p in picks]

    lidar_elevs, lidar_masks = [], []
    s1_stacks = []
    n_bands = None

    for pid in pids:
        lid_path = os.path.join(lidar_patches_dir, f"lidar_patch_{pid}.tif")
        with rasterio.open(lid_path) as src:
            lidar_elevs.append(src.read(1))
            lidar_masks.append(src.read(2))

        s1_dir = os.path.join(out_s1_dir, f"s1_patch_{pid}")
        timestep_stack = []
        for tpath in sorted(glob_module.glob(os.path.join(s1_dir, "t*.tif"))):
            with rasterio.open(tpath) as src:
                arr = src.read()
            if n_bands is None:
                n_bands = arr.shape[0]
            if use_db:
                with np.errstate(divide="ignore", invalid="ignore"):
                    arr = 10 * np.log10(np.where(arr > 0, arr, np.nan))
            timestep_stack.append(arr)
        s1_stacks.append(timestep_stack)

    lidar_anoms = []
    for e, m in zip(lidar_elevs, lidar_masks):
        valid = m.astype(bool)
        patch_mean = e[valid].mean() if valid.any() else e.mean()
        lidar_anoms.append(e - patch_mean)

    valid_anom_vals = np.concatenate([
        a[m.astype(bool)] for a, m in zip(lidar_anoms, lidar_masks) if m.any()
    ]) if any(m.any() for m in lidar_masks) else np.concatenate(lidar_anoms).ravel()
    lo, hi = np.nanpercentile(valid_anom_vals, [2, 98])
    elev_abs_max = max(abs(lo), abs(hi))
    elev_vmin, elev_vmax = -elev_abs_max, elev_abs_max

    band_vmin, band_vmax = [], []
    for b in range(n_bands):
        vals = np.concatenate([ts[b].ravel() for stack in s1_stacks for ts in stack])
        vmin, vmax = np.nanpercentile(vals, [2, 98])
        band_vmin.append(vmin)
        band_vmax.append(vmax)

    labels = list(pol_labels[:n_bands]) if n_bands <= len(pol_labels) else [f"band{b+1}" for b in range(n_bands)]

    n_timesteps = len(s1_stacks[0])
    n_cols = 2 + n_timesteps * n_bands

    fig, axes = plt.subplots(n_samples, n_cols, figsize=(4.2 * n_cols, 4 * n_samples))
    if n_samples == 1:
        axes = axes[None, :]

    for row, pid in enumerate(pids):
        im_elev = axes[row, 0].imshow(lidar_anoms[row], cmap="RdBu_r", vmin=elev_vmin, vmax=elev_vmax)
        axes[row, 0].set_title(f"LiDAR Elev anomaly (patch {pid})", fontsize=9)
        axes[row, 0].axis("off")
        fig.colorbar(im_elev, ax=axes[row, 0], fraction=0.046, pad=0.04, label="Elevation - patch mean (m)")

        axes[row, 1].imshow(lidar_masks[row], cmap="gray", vmin=0, vmax=1)
        axes[row, 1].set_title("LiDAR Valid Mask", fontsize=9)
        axes[row, 1].axis("off")

        col = 2
        for ti, arr in enumerate(s1_stacks[row]):
            for b in range(n_bands):
                im_b = axes[row, col].imshow(arr[b], cmap="gray", vmin=band_vmin[b], vmax=band_vmax[b])
                axes[row, col].set_title(f"S1 {labels[b]} t{ti}" + (" (dB)" if use_db else ""), fontsize=9)
                axes[row, col].axis("off")
                fig.colorbar(im_b, ax=axes[row, col], fraction=0.046, pad=0.04,
                             label="Sigma0 (dB)" if use_db else "Sigma0 (linear)")
                col += 1

    plt.tight_layout()
    plt.show()

show_multiple_matched_patches(lidar_patches_tessa_dir, out_s1_dir, n_samples=5)

## 12. Quantitative collocation check -- `compute_collocation_quality_stats` (Michel's code, unchanged)

Rather than judging alignment by eye, this checks whether rougher LiDAR terrain actually correlates with higher/more variable Sentinel-1 backscatter across patches -- a real physical relationship (rougher surfaces scatter more diffusely). If patches are correctly spatially matched, this correlation should be positive and visible. If collocation were off (e.g. patches randomly paired with the wrong Sentinel-1 window), this correlation should be weak or absent.

In [ ]:
def compute_collocation_quality_stats(lidar_patches_dir, out_s1_dir, timestep=0, pol_labels=("VV", "VH")):
    sample_dirs = sorted(glob_module.glob(os.path.join(out_s1_dir, "s1_patch_*")))
    roughness_vals, pids = [], []
    band_mean_vals = band_std_vals = None
    n_bands = None

    for s1_dir in sample_dirs:
        pid = os.path.basename(s1_dir).split("_")[-1]
        lid_path = os.path.join(lidar_patches_dir, f"lidar_patch_{pid}.tif")
        if not os.path.exists(lid_path):
            continue
        with rasterio.open(lid_path) as src:
            elev = src.read(1)
            mask = src.read(2).astype(bool)
        if mask.sum() < 10:
            continue

        t_path = os.path.join(s1_dir, f"t{timestep}.tif")
        if not os.path.exists(t_path):
            continue
        with rasterio.open(t_path) as src:
            arr = src.read()
        if n_bands is None:
            n_bands = arr.shape[0]
            band_mean_vals = [[] for _ in range(n_bands)]
            band_std_vals = [[] for _ in range(n_bands)]

        with np.errstate(divide="ignore", invalid="ignore"):
            arr_db = 10 * np.log10(np.where(arr > 0, arr, np.nan))
        if np.all(np.isnan(arr_db)):
            continue

        pids.append(pid)
        roughness_vals.append(float(np.std(elev[mask])))
        for b in range(n_bands):
            band_mean_vals[b].append(float(np.nanmean(arr_db[b])))
            band_std_vals[b].append(float(np.nanstd(arr_db[b])))

    roughness_vals = np.array(roughness_vals)
    labels = list(pol_labels[:n_bands]) if n_bands <= len(pol_labels) else [f"band{b+1}" for b in range(n_bands)]
    print(f"Computed stats for {len(pids)} patches")

    fig, axes = plt.subplots(n_bands, 2, figsize=(12, 5 * n_bands), squeeze=False)
    corr_results = {}
    for b in range(n_bands):
        mean_vals = np.array(band_mean_vals[b])
        std_vals = np.array(band_std_vals[b])
        corr_mean = np.corrcoef(roughness_vals, mean_vals)[0, 1]
        corr_std = np.corrcoef(roughness_vals, std_vals)[0, 1]
        corr_results[labels[b]] = (corr_mean, corr_std)
        print(f"[{labels[b]}] Correlation: LiDAR roughness vs mean (dB): {corr_mean:.3f}")
        print(f"[{labels[b]}] Correlation: LiDAR roughness vs std (dB):  {corr_std:.3f}")

        axes[b, 0].scatter(roughness_vals, mean_vals, s=8, alpha=0.4)
        axes[b, 0].set_xlabel("LiDAR elevation std (roughness)")
        axes[b, 0].set_ylabel(f"S1 {labels[b]} mean backscatter (dB)")
        axes[b, 0].set_title(f"{labels[b]}: r = {corr_mean:.3f}")

        axes[b, 1].scatter(roughness_vals, std_vals, s=8, alpha=0.4)
        axes[b, 1].set_xlabel("LiDAR elevation std (roughness)")
        axes[b, 1].set_ylabel(f"S1 {labels[b]} std backscatter (dB)")
        axes[b, 1].set_title(f"{labels[b]}: r = {corr_std:.3f}")

    plt.tight_layout()
    plt.show()

    return pids, roughness_vals, band_mean_vals, band_std_vals, corr_results

_ = compute_collocation_quality_stats(lidar_patches_tessa_dir, out_s1_dir)

## 13. (Optional) Save a labeled sample patch figure -- `save_sample_patch_figure` (Michel's code, unchanged)

Reads each timestep's real acquisition date from the patch's own `attrs.json` and saves a figure matching the style of the original Sentinel-2 QC image.

**Caveat:** since `attrs_paths` was set to `None` for every product in Step 6 (this project's raw SAFE downloads don't carry a metadata sidecar matching Michel's `s1_safe_N_metadata.json` naming convention -- see the earlier note on `acquisition_date: null`), every panel below will currently show "unknown date" instead of a real date. Fix the metadata-attaching step first if dated labels matter for a figure you intend to use externally.

In [ ]:
def save_sample_patch_figure(patch_id, lidar_patches_dir, out_s1_dir, region_name, lidar_date,
                              out_path=None, use_db=True):
    lid_path = os.path.join(lidar_patches_dir, f"lidar_patch_{patch_id}.tif")
    s1_dir = os.path.join(out_s1_dir, f"s1_patch_{patch_id}")

    with rasterio.open(lid_path) as src:
        lidar_elev = src.read(1)
        lidar_mask = src.read(2)

    attrs_path = os.path.join(s1_dir, "attrs.json")
    attrs_list = json.load(open(attrs_path)) if os.path.exists(attrs_path) else []

    def fmt_date(attrs):
        if not attrs or not attrs.get("acquisition_date"):
            return "unknown date"
        y, m, d = attrs["acquisition_date"].split("-")
        return f"{d}-{m}-{y[2:]}"

    s1_timesteps = sorted(glob_module.glob(os.path.join(s1_dir, "t*.tif")))
    n = len(s1_timesteps)

    fig, axes = plt.subplots(1, n + 2, figsize=(4 * (n + 2), 4.5))
    fig.suptitle("Sample Training Patch Set", fontsize=16, fontweight="bold")

    axes[0].imshow(lidar_elev, cmap="viridis")
    axes[0].set_title(f"LiDAR ({region_name}, {lidar_date})", fontsize=10, fontweight="bold")
    axes[0].axis("off")

    axes[1].imshow(lidar_mask, cmap="gray")
    axes[1].set_title("LiDAR Validity Mask", fontsize=10, fontweight="bold")
    axes[1].axis("off")

    for i, tpath in enumerate(s1_timesteps):
        with rasterio.open(tpath) as src:
            vv = src.read(1)
        if use_db:
            with np.errstate(divide="ignore", invalid="ignore"):
                vv = 10 * np.log10(np.where(vv > 0, vv, np.nan))
        date_str = fmt_date(attrs_list[i]) if i < len(attrs_list) else "unknown date"
        axes[2 + i].imshow(vv, cmap="gray", vmin=np.nanpercentile(vv, 2), vmax=np.nanpercentile(vv, 98))
        axes[2 + i].set_title(f"Sentinel-1: {date_str}", fontsize=10, fontweight="bold")
        axes[2 + i].axis("off")

    plt.tight_layout()
    if out_path:
        plt.savefig(out_path, dpi=150, bbox_inches="tight")
        print(f"Saved: {out_path}")
    plt.show()


sample = sorted(glob_module.glob(os.path.join(out_s1_dir, "s1_patch_*")))
if sample:
    pid = os.path.basename(sample[0]).split("_")[-1]
    out_png = os.path.join(REPO_DIR, "input_data", f"sample_patch_{REGION}_tessa_utm8n_{pid}.png")
    save_sample_patch_figure(pid, lidar_patches_tessa_dir, out_s1_dir, region_name=REGION.title(),
                              lidar_date="16-04-24", out_path=out_png)